# Beer PCA 1대1 예측 실험 (화학 X → 전문가 관능 Y)

- 목적: **PCA(주성분분석, Principal Component Analysis)** 로 화학변수(X)를 선형 특징추출(feature extraction)하여 차원을 줄였을 때,
  관능평가(Y) 예측 성능이 어떻게 변하는지 확인.
- 분할: 논문과 동일하게 **70/30(175/75)** + **맥주 스타일 기준 층화(stratified split)**.
- 캐시: PCA 변환 결과(전체 컴포넌트)를 `.npy/.joblib`로 저장해서 이후 재사용.

> 주의: `sklearn`의 PCA/GBR는 **CPU 연산**이므로 A100 GPU는 거의 사용하지 않습니다. (Autoencoder 실험부터 GPU 사용)


In [ ]:
# (선택) 필요한 패키지 설치
# 서버 환경에 이미 설치되어 있다면 이 셀은 건너뛰어도 됩니다.
# 인터넷이 막혀 있으면 pip가 실패할 수 있습니다.

!pip -q install pandas numpy scikit-learn openpyxl joblib tqdm matplotlib


In [ ]:
import os, json, re, time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

from joblib import dump, load
from tqdm.auto import tqdm
from IPython.display import display

# =========================
# 사용자 설정(여기만 수정)
# =========================
# 1) 데이터 경로: 엑셀(xlsx) 또는 S1/S4 CSV가 있는 폴더
DATA_ROOT = Path("./data")  # 예: Path("/home/your_id/beer/data")

# 2) 엑셀 파일 경로(없으면 None으로 두고, DATA_ROOT 내 CSV를 찾음)
XLSX_PATH = DATA_ROOT / "Supplemental Files and Figure source files.xlsx"  # 파일명 맞게 수정
USE_XLSX = XLSX_PATH.exists()

# 3) 결과/캐시 저장 폴더
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTDIR = Path("./runs") / f"pca_1to1_{RUN_TAG}"
OUTDIR.mkdir(parents=True, exist_ok=True)

# 4) 논문과 동일 split
TEST_SIZE = 0.30
RANDOM_STATE = 0  # 논문/Zenodo 코드와 동일하게 0 사용

# 5) PCA 컴포넌트 수 후보 (빠른 버전 / 전체 버전 선택)
# - full: 1..max_k 전체 sweep (시간 더 걸림)
# - quick: 대표 k만 sweep
K_SWEEP_MODE = "quick"  # "quick" or "full"

# 6) GBR(Gradient Boosting Regressor) 하이퍼파라미터
# - 논문은 GridSearchCV를 사용했지만, 1대1 전수에서는 계산량이 커서 우선 고정값으로 진행 권장.
GBR_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    loss="squared_error",
    random_state=RANDOM_STATE,
)

# 병렬화는 여기서는 사용하지 않음(간단/안정 우선).
# CPU 코어가 많으면 joblib.Parallel로 target별 병렬화 가능.

print("OUTDIR =", OUTDIR)


In [ ]:
def _detect_sheet_by_columns(xlsx_path: Path, required_cols):
    """엑셀에서 required_cols를 모두 포함하는 시트를 찾아 반환"""
    xls = pd.ExcelFile(xlsx_path)
    for s in xls.sheet_names:
        try:
            cols = pd.read_excel(xlsx_path, sheet_name=s, nrows=0).columns
            if all(c in cols for c in required_cols):
                return s
        except Exception:
            continue
    return None

def load_beer_data(data_root: Path, xlsx_path: Path = None):
    """S1(화학+메타)와 S4(전문가 관능) 데이터를 로드하여 합친 DF 반환.

    우선순위:
    1) data_root에 'Supplemental File S1.csv' & 'Supplemental File S4.csv'가 있으면 CSV 사용
    2) 아니면 xlsx_path에서 시트를 자동 탐지해서 로드
    """
    s1_csv = data_root / "Supplemental File S1.csv"
    s4_csv = data_root / "Supplemental File S4.csv"

    if s1_csv.exists() and s4_csv.exists():
        df_s1 = pd.read_csv(s1_csv)
        df_s4 = pd.read_csv(s4_csv)
    else:
        if xlsx_path is None or (not Path(xlsx_path).exists()):
            raise FileNotFoundError(
                "CSV도 없고 XLSX도 못 찾았습니다. DATA_ROOT/XLSX_PATH를 확인하세요."
            )
        xlsx_path = Path(xlsx_path)

        # S1: 화학 데이터는 acetaldehyde ~ sulfur_sum 컬럼을 포함
        sheet_s1 = _detect_sheet_by_columns(xlsx_path, required_cols=["acetaldehyde", "sulfur_sum"])
        # S4: 관능 데이터는 A_malt_all ~ overall 컬럼을 포함
        sheet_s4 = _detect_sheet_by_columns(xlsx_path, required_cols=["A_malt_all", "overall"])

        if sheet_s1 is None or sheet_s4 is None:
            xls = pd.ExcelFile(xlsx_path)
            raise RuntimeError(
                f"시트 자동탐지 실패. 엑셀 시트명 목록: {xls.sheet_names}\n"
                "sheet_s1/sheet_s4를 직접 지정하도록 코드를 수정해야 할 수 있습니다."
            )

        df_s1 = pd.read_excel(xlsx_path, sheet_name=sheet_s1)
        df_s4 = pd.read_excel(xlsx_path, sheet_name=sheet_s4)

    # 공통 키: beer 컬럼이 있는 경우가 많음
    if "beer" in df_s1.columns:
        df_s1 = df_s1.set_index("beer")
    if "beer" in df_s4.columns:
        df_s4 = df_s4.set_index("beer")

    df = df_s1.join(df_s4, how="inner")

    return df

df = load_beer_data(DATA_ROOT, XLSX_PATH if USE_XLSX else None)
print("Loaded DF shape:", df.shape)
print("Columns example:", list(df.columns)[:10])


In [ ]:
# ====== X / Y / style label 컬럼 설정 ======
# X(화학): acetaldehyde ~ sulfur_sum (231개)
if "acetaldehyde" not in df.columns or "sulfur_sum" not in df.columns:
    raise KeyError("화학 컬럼 범위(acetaldehyde~sulfur_sum)를 찾지 못했습니다. 컬럼명을 확인하세요.")
X_df = df.loc[:, "acetaldehyde":"sulfur_sum"].copy()

# Y(관능): A_malt_all ~ overall (50개)
if "A_malt_all" not in df.columns or "overall" not in df.columns:
    raise KeyError("관능 컬럼 범위(A_malt_all~overall)를 찾지 못했습니다. 컬럼명을 확인하세요.")
Y_df = df.loc[:, "A_malt_all":"overall"].copy()

# stratify label: tasting_category_fine (논문/Zenodo 기준)
# 일부 파일에서는 tasting_category_fine_chem 같은 이름일 수 있어 fallback 처리
if "tasting_category_fine" in df.columns:
    style = df["tasting_category_fine"].astype(str)
elif "tasting_category_fine_chem" in df.columns:
    style = df["tasting_category_fine_chem"].astype(str)
else:
    # tasting_category 관련 컬럼 후보 출력
    cand = [c for c in df.columns if "tasting_category" in c]
    raise KeyError(f"스타일 라벨 컬럼을 못 찾았습니다. 후보: {cand}")

print("X shape:", X_df.shape, "Y shape:", Y_df.shape, "style #unique:", style.nunique())

# ====== stratified split (175/75) ======
X_train_df, X_test_df, Y_train_df, Y_test_df, style_train, style_test = train_test_split(
    X_df, Y_df, style,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=style
)

print("Train/Test:", X_train_df.shape, X_test_df.shape)

# 스타일 분포 확인
train_counts = style_train.value_counts().sort_index()
test_counts  = style_test.value_counts().sort_index()
dist = pd.DataFrame({"train": train_counts, "test": test_counts}).fillna(0).astype(int)
dist["train_ratio"] = dist["train"] / dist["train"].sum()
dist["test_ratio"]  = dist["test"] / dist["test"].sum()
dist["abs_diff"] = (dist["train_ratio"] - dist["test_ratio"]).abs()
dist = dist.sort_values("abs_diff", ascending=False)

print("=== style stratification check (top diff) ===")
display(dist.head(10))


In [ ]:
# ====== PCA 캐시 ======
# 캐시는 'runs'와 분리해서 project-level ./cache 아래에 저장합니다.
# (같은 split이면 다음 실험에서 재사용 가능)

CACHE_ROOT = Path("./cache")
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

CACHE_DIR = CACHE_ROOT / f"pca_expertpanel_seed{RANDOM_STATE}_test{TEST_SIZE}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

meta_path = CACHE_DIR / "meta.json"
z_train_path = CACHE_DIR / "Z_train_full.npy"
z_test_path  = CACHE_DIR / "Z_test_full.npy"
z_all_path   = CACHE_DIR / "Z_all_full.npy"
pca_path     = CACHE_DIR / "pca.joblib"
imputer_path = CACHE_DIR / "imputer.joblib"
scaler_path  = CACHE_DIR / "scaler.joblib"

# 최대 PC 수: min(n_train-1, d)
max_k = min(X_train_df.shape[0] - 1, X_train_df.shape[1])
print("max_k (max PCA components) =", max_k)
print("CACHE_DIR =", CACHE_DIR)

def compute_and_cache_pca():
    # 1) mean imputation (train 기준)
    imputer = SimpleImputer(strategy="mean")
    X_train_imp = imputer.fit_transform(X_train_df.values)
    X_test_imp  = imputer.transform(X_test_df.values)
    X_all_imp   = imputer.transform(X_df.values)

    # 2) standardization (train 기준)
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train_imp)
    X_test_sc  = scaler.transform(X_test_imp)
    X_all_sc   = scaler.transform(X_all_imp)

    # 3) PCA (train 기준)
    pca = PCA(n_components=max_k, svd_solver="full", random_state=RANDOM_STATE)
    Z_train = pca.fit_transform(X_train_sc)
    Z_test  = pca.transform(X_test_sc)
    Z_all   = pca.transform(X_all_sc)

    # save
    np.save(z_train_path, Z_train)
    np.save(z_test_path, Z_test)
    np.save(z_all_path, Z_all)
    dump(pca, pca_path)
    dump(imputer, imputer_path)
    dump(scaler, scaler_path)

    meta = {
        "random_state": RANDOM_STATE,
        "test_size": TEST_SIZE,
        "max_k": int(max_k),
        "n_train": int(X_train_df.shape[0]),
        "n_test": int(X_test_df.shape[0]),
        "n_features": int(X_train_df.shape[1]),
        "X_columns": list(X_train_df.columns),
        "Y_columns": list(Y_train_df.columns),
        "train_index": list(X_train_df.index.astype(str)),
        "test_index": list(X_test_df.index.astype(str)),
        "style_train_counts": style_train.value_counts().to_dict(),
        "style_test_counts": style_test.value_counts().to_dict(),
        "explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
    }
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    return Z_train, Z_test, Z_all, pca, imputer, scaler

if z_train_path.exists() and z_test_path.exists() and pca_path.exists() and imputer_path.exists() and scaler_path.exists() and meta_path.exists():
    print("✅ PCA cache found. Loading...")
    Z_train_full = np.load(z_train_path)
    Z_test_full  = np.load(z_test_path)
    Z_all_full   = np.load(z_all_path)
    pca = load(pca_path)
    imputer = load(imputer_path)
    scaler = load(scaler_path)
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
else:
    print("⏳ PCA cache not found. Computing...")
    Z_train_full, Z_test_full, Z_all_full, pca, imputer, scaler = compute_and_cache_pca()

print("Z_train_full:", Z_train_full.shape, "Z_test_full:", Z_test_full.shape)
print("Explained variance (cum, first 10):", np.cumsum(pca.explained_variance_ratio_)[:10])


In [ ]:
# ====== k 리스트 설정 ======
if K_SWEEP_MODE == "full":
    K_LIST = list(range(1, max_k + 1))
else:
    # quick: 대표 k + max_k
    K_LIST = [1, 2, 4, 8, 16, 32, 64, 128, max_k]
    K_LIST = sorted(list(set([k for k in K_LIST if k <= max_k])))

print("K_LIST:", K_LIST)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def fit_eval_one_target(y_name: str, k: int):
    # 1) 데이터 슬라이싱
    Xtr = Z_train_full[:, :k]
    Xte = Z_test_full[:, :k]
    ytr = Y_train_df[y_name].values
    yte = Y_test_df[y_name].values

    # 2) 모델
    model = GradientBoostingRegressor(**GBR_PARAMS)
    model.fit(Xtr, ytr)

    # 3) 평가
    pred_tr = model.predict(Xtr)
    pred_te = model.predict(Xte)

    out = {
        "target": y_name,
        "k": k,
        "r2_train": float(r2_score(ytr, pred_tr)),
        "r2_test": float(r2_score(yte, pred_te)),
        "rmse_train": rmse(ytr, pred_tr),
        "rmse_test": rmse(yte, pred_te),
    }
    return out

results = []
targets = list(Y_train_df.columns)

for k in tqdm(K_LIST, desc="k sweep"):
    for y_name in targets:
        results.append(fit_eval_one_target(y_name, k))

res_df = pd.DataFrame(results)
res_path = OUTDIR / "pca_gbr_1to1_results.csv"
res_df.to_csv(res_path, index=False)
print("Saved:", res_path)

# 요약: k별 평균 성능
summary = res_df.groupby("k").agg(
    r2_test_mean=("r2_test", "mean"),
    r2_test_median=("r2_test", "median"),
    rmse_test_mean=("rmse_test", "mean"),
).reset_index().sort_values("r2_test_mean", ascending=False)

summary_path = OUTDIR / "pca_gbr_1to1_summary_by_k.csv"
summary.to_csv(summary_path, index=False)
print("Saved:", summary_path)

display(summary.head(10))


In [ ]:
import matplotlib.pyplot as plt

# k별 평균 R2 추이
plt.figure()
plt.plot(summary.sort_values("k")["k"], summary.sort_values("k")["r2_test_mean"], marker="o")
plt.xlabel("k (number of PCA components)")
plt.ylabel("mean test R^2 across 50 sensory targets")
plt.title("PCA(k) + GBR : mean test R^2")
plt.grid(True)
plt.show()

# best k에서 target별 top/bottom
best_k = int(summary.iloc[0]["k"])
sub = res_df[res_df["k"] == best_k].sort_values("r2_test", ascending=False)
print("best_k =", best_k)
display(sub.head(10))
display(sub.tail(10))


## 다음 단계(권장)

1) **데이터 누수(data leakage) 방지**  
- `K_LIST`나 `GBR_PARAMS`를 튜닝할 때는, test를 보면서 고르면 편향이 생깁니다.  
- 논문용으로는 train 안에서 `train/val` 또는 `CV`로 선택하고 test는 마지막 1회만 확인하는 구조가 안전합니다.

2) Autoencoder(오토인코더, Autoencoder) 실험  
- PCA와 동일 split/전처리를 유지한 상태에서  
- AE latent 차원(k)을 바꿔가며 동일 평가를 하면, “선형 vs 비선형 특징추출” 비교가 깔끔해집니다.

3) 캐시 재사용  
- 이 노트북은 `runs/.../cache_pca/`에 PCA 변환결과를 저장합니다.  
- 동일 split이라면 이후 실험에서 `Z_train_full.npy`를 로드해서 재사용 가능합니다.
